In [6]:
import os
import pandas as pd
from dataclasses import dataclass
from notebooks._utils import calculate_accuracy

data_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"
dataset_names = ["hotpot-reasoning-debug"]

@dataclass
class Args:
    num_paraphrases: int
    repeat: int
    method: str = "per_prompt"
    temperature: float = 1.0
    top_p: float = 0.95
    max_new_tokens: int = 2048
    reasoning: bool = True
    num_fewshots: int = 0
    additional_paraphrases_file: str = None

@dataclass
class DummyDataset:
    dataset_root:str

In [16]:
from utils import get_baseline_dumpfile

        
# for model_name in ["qwen3_1.7b", "qwen3_4b", "qwen3_8b", "qwen3_14b", "qwen3_32b"]:
for model_name in ["qwen3_1.7b"]:
    for dataset_name in dataset_names:
        dataset_path = os.path.join(data_root, "hotpot-reasoning-debug", model_name)
        dataset = DummyDataset(dataset_root=dataset_path)
        
        baseline_args = Args(num_paraphrases=0, repeat=4)
        paraphrase_args = Args(num_paraphrases=1, repeat=2)

        for args in [baseline_args, paraphrase_args]:    
            filename = get_baseline_dumpfile(dataset, args)
            
            print(f"=================== {args.num_paraphrases + 1} prompt x {args.repeat} thinkings =====================")
                
            print(f"Dataset: {dataset_name.split('-')[0]}, Model: {model_name}")
                
            if os.path.exists(filename) is False:
                print(f"Baseline file {filename} does not exist.")
            else:
                baseline_df = pd.read_feather(filename)
                baseline_df = baseline_df[baseline_df['thinking'].str.len() > 0]
                baseline_acc = calculate_accuracy(
                    baseline_df, is_multichoice=False,
                    birdirect=True, use_generation=True, by_probs=False,
                    label=f"Average accuracy")

                # Simple logits-level ensemble
                ensemble_filename = os.path.join(
                    filename.replace(".feather", ""), 
                    "hotpot.logits.avg.0fshots.1samples.-1paras.feather"
                )
                if os.path.exists(ensemble_filename):
                    ensemble_df = pd.read_feather(ensemble_filename)
                    ensemble_acc = calculate_accuracy(
                        ensemble_df, is_multichoice=False,
                        birdirect=True, use_generation=True, by_probs=False,
                        label="Logits-level Ensembled accuracy")
                else:
                    print(f"Ensemble file {ensemble_filename} does not exist.")
                
                # Middle-layer ensemble
                logits_ensemble_filename = os.path.join(
                    filename.replace(".feather", ""), 
                    "hotpot.logits.avg.avglayer.layer21.alpha100.token-last.multilayer.0fshots.1samples.-1paras.feather"
                )
                
                if os.path.exists(logits_ensemble_filename):
                    ensemble_df = pd.read_feather(logits_ensemble_filename)
                    ensemble_acc = calculate_accuracy(
                        ensemble_df, is_multichoice=False,
                        birdirect=True, use_generation=True, by_probs=False,
                        label="Logits-level Ensembled accuracy")
                else:
                    print(f"Ensemble file {logits_ensemble_filename} does not exist.")

=================== 1 prompt x 4 thinkings =====================
Dataset: hotpot, Model: qwen3_1.7b
Acc: 0.2195 ==> 🏷️ Average accuracy
Acc: 0.2100 ==> 🏷️ Logits-level Ensembled accuracy
Acc: 0.2100 ==> 🏷️ Logits-level Ensembled accuracy
=================== 2 prompt x 2 thinkings =====================
Dataset: hotpot, Model: qwen3_1.7b
Acc: 0.2163 ==> 🏷️ Average accuracy
Acc: 0.2050 ==> 🏷️ Logits-level Ensembled accuracy
Ensemble file /home/xzhao/workspace/GYB_self-ensemble/datasets/hotpot-reasoning-debug/qwen3_1.7b/baseline_per_prompt.0shots.1paras.temp1.0.topp0.95.maxnew2048.repeat2/hotpot.logits.avg.avglayer.layer21.alpha100.token-last.multilayer.0fshots.1samples.-1paras.feather does not exist.


In [3]:
filename

'/home/xzhao/workspace/GYB_self-ensemble/datasets/hotpot-reasoning-debug/qwen3_32b/baseline_per_prompt.0shots.1paras.temp1.0.topp0.95.maxnew2048.repeat2.selfparas.feather'

In [8]:
ensemble_df[ensemble_df['uuid']=="5a7a06935542990198eaf050"]

,uuid,answers,prediction,generation,paraphrases,is_orig,prompts,labels,label_probs,generation_lemmas,answer_lemmas,correct
26,5a7a06935542990198eaf050,[Arthur's Magazine],!!!!!!!!!!!!!!!!!!!!,!!!!!!!!!!!!!!!!!!!!,[<|im_start|>system\nThink through the given q...,"[True, True, False, False]","[Think through the given question, then output...",[],[],"[!, !, !, !, !, !, !, !, !, !, !, !, !, !, !, ...","[[arthur, 's, magazine]]",0


In [10]:
paraphrases = [
"<|im_start|>system\nThink through the given question, then output only the final answer.\nOutput format (strict): ### Answer: <final answer>\nDo not include any explanation, reasoning, citations, or extra text—only the single answer line.<|im_end|>\n<|im_start|>user\nWhich magazine was started first Arthur's Magazine or First for Women?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, I need to figure out which magazine was started first between Arthur's Magazine and First for Women. Let me start by recalling what I know about these publications.\n\nFirst, Arthur's Magazine. I think there might be a few magazines with similar names. There's The Atlantic Monthly, which was founded in 1857, but that's not Arthur's. Wait, maybe it's Arthur's Magazine from the 19th century? Let me check my memory. I remember that there was an Arthur's Magazine in the 1840s or 50s. Maybe it was a literary or general interest magazine. Let me try to remember. I think Arthur's Magazine was published in the mid-1800s. Perhaps around 1844? Or was that another magazine?\n\nNow, First for Women. That sounds more modern. I think it's a women's magazine that started in the late 20th century. Maybe the 1980s or 1990s. It's targeted at women, focusing on lifestyle, fashion, and advice. Let me try to recall the exact launch date. If I remember correctly, First for Women was launched in 1989. \n\nSo if Arthur's Magazine was founded in the 1840s or 50s, and First for Women in 1989, then Arthur's Magazine was started way earlier. But I need to confirm the exact dates. Let me think again. Arthur's Magazine, also known as Arthur's, was indeed founded in 1844 by John L. Sullivan. It was a monthly magazine that featured various articles, literature, and was quite popular in the 19th century. It had a long run, maybe until the 1950s or 60s. \n\nOn the other hand, First for Women was indeed started much later. According to my knowledge, it was launched in 1989 by The Meredith Corporation. So there's a gap of over 140 years between the two. Therefore, Arthur's Magazine was started first.\n</think>",
 '<|im_start|>system\nThink through the given question, then output only the final answer.\nOutput format (strict): ### Answer: <final answer>\nDo not include any explanation, reasoning, citations, or extra text—only the single answer line.<|im_end|>\n<|im_start|>user\nWhich magazine was started first Arthur\'s Magazine or First for Women?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, I need to figure out which magazine was started first between Arthur\'s Magazine and First for Women. Let me start by recalling what I know about these publications.\n\nFirst, Arthur\'s Magazine. I think there might be a few magazines with "Arthur" in the name. There\'s Arthur magazine, which I believe was a literary and cultural magazine. I\'m not exactly sure when it was founded. Maybe in the 20th century? Let me try to remember or think of key events. Wait, there was an Arthur\'s Magazine in the 19th century, perhaps? Or maybe it\'s a more recent publication. I need to check my memory here.\n\nThen there\'s First for Women. I know that First for Women is a magazine aimed at women, focusing on lifestyle, fashion, and personal stories. I think it\'s a more modern magazine. When was it launched? I believe it was started in the late 20th century. Maybe around the 1980s or 1990s? Let me see. I recall that First for Women was launched in the United States. I might have heard that it was started by the Conde Nast Publications, but I\'m not certain. Alternatively, maybe it\'s a different publisher. \n\nNow, cross-referencing the dates. If First for Women was started in the 90s, and Arthur\'s Magazine was perhaps earlier, then Arthur\'s would be first. But I need to get the exact years. Let me think. Arthur\'s Magazine: maybe it\'s the one that was published in the 19th century? There was a publication called "Arthur\'s Home Magazine" or something similar. Wait, there was a magazine called "Arthur\'s Magazine" that started in the 1840s or 1850s. For example, I remember that "Arthur\'s Home Magazine" was a 19th-century publication that was aimed at families and had stories and articles. So if that\'s the case, then Arthur\'s Magazine would have been founded much earlier than First for Women.\n\nAlternatively, maybe the modern Arthur\'s Magazine is different. Let me confirm. Suppose Arthur\'s Magazine, as in the one that\'s a contemporary literary magazine, but that might be a different entity. For example, the current Arthur magazine was founded in 1997. Wait, but if that\'s the case, then First for Women, which was founded in 1989, would be earlier. But that would conflict with the historical Arthur\'s Magazine from the 19th century. So there might be two different Arthur\'s Magazines. \n\nWait, the question is about Arthur\'s Magazine versus First for Women. The key is to know which one was started first. If Arthur\'s Magazine in question is the 19th-century one, then it\'s way before First for Women. However, if the Arthur\'s Magazine in question is the modern one (founded in 1997), then First for Women would be older. So I need to determine which Arthur\'s Magazine is being referred to here. \n\nSince the question is in the present day context, maybe the user is referring to the modern Arthur\'s Magazine. But I\'m not sure. Let me check my knowledge. The Arthur magazine, which is a literary quarterly, was founded in 1997 by the family of writer David Foster Wallace. So that\'s the modern Arthur\'s. If that\'s the case, then First for Women was started in 1989, which is earlier. Therefore, First for Women would be first. However, if the question is about the 19th-century Arthur\'s Home Magazine, which was published from the 1840s to the 1930s, then that one would be much earlier. \n\nBut the problem is that the name "Arthur\'s Magazine" could be ambiguous. However, given that First for Women is a well-known magazine that was launched in 1989, and assuming that the Arthur\'s Magazine being referred to is the one from the 19th century (Arthur\'s Home Magazine), then Arthur\'s Magazine was started first. But if it\'s the modern Arthur\'s, then First for Women is older. \n\nSo, to answer this, I need to determine which Arthur\'s Magazine is in question. Since the user is asking about Arthur\'s Magazine versus First for Women, it\'s possible that the intended comparison is between the original Arthur\'s and First for Women. Given that First for Women was started in 1989, and Arthur\'s Home Magazine started in 1844, Arthur\'s Magazine was first. Therefore, Arthur\'s Magazine was started first.\n</think>',
 '<|im_start|>system\nThink through the given question, then output only the final answer.\nOutput format (strict): ### Answer: <final answer>\nDo not include any explanation, reasoning, citations, or extra text—only the single answer line.<|im_end|>\n<|im_start|>user\nWhich magazine began publication earlier: Arthur\'s Magazine or First for Women?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let me try to figure out which magazine was published earlier between Arthur\'s Magazine and First for Women.\n\nFirst, I need to recall when these magazines were launched. I remember that First for Women is a more recent publication, maybe from the 1980s or 1990s. It\'s a women\'s magazine that focuses on lifestyle, health, and news. On the other hand, Arthur\'s Magazine sounds a bit older. Arthur\'s might have been around in the late 1800s or early 1900s. Wait, there was an Arthur\'s Magazine in the 19th century. Let me check my memory. Arthur\'s Magazine was published by the publisher of "Arthur\'s Home Magazine," which was around the 1840s. Wait, no, maybe that\'s a different one. Let me think again. There was a publication called "Arthur\'s Magazine" that started in the 1800s. First for Women, I think, started in the 1980s. So Arthur\'s Magazine would be earlier. Wait, but I should be careful. Maybe Arthur\'s Magazine was a more recent revival? No, I think the original Arthur\'s Magazine was from the 1800s. Let me confirm. Arthur\'s Magazine was first published in 1863. First for Women started in 1989. So the answer is Arthur\'s Magazine.\n</think>',
 '<|im_start|>system\nThink through the given question, then output only the final answer.\nOutput format (strict): ### Answer: <final answer>\nDo not include any explanation, reasoning, citations, or extra text—only the single answer line.<|im_end|>\n<|im_start|>user\nWhich magazine began publication earlier: Arthur\'s Magazine or First for Women?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let\'s tackle this question. So, the user is asking which magazine started earlier between Arthur\'s Magazine and First for Women.\n\nFirst, I need to figure out the publication dates of both magazines. Let me start with Arthur\'s Magazine. I remember that there are a few magazines with similar names. The Arthur\'s Magazine that comes to mind is the one published by DC Comics in the 1970s. But wait, maybe there\'s another one? Let me double-check. Yes, Arthur\'s Magazine was a monthly humor magazine launched by DC Comics. I think it was around 1977. I remember it had a lot of satirical content and some iconic covers, like the "Superman vs. Santa Claus" issue. But I should verify the exact year. If it\'s 1977, that\'s definitely in the late 1970s.\n\nNow, First for Women. I\'m less familiar with the launch date of this one. From what I recall, First for Women is a women\'s magazine that focuses on real-life stories, often with a focus on empowerment and everyday experiences. I think it started in the 1980s. Wait, when exactly? Let me think. I believe it was launched in 1989. It was created as a response to the changing times in the 80s, targeting working women who wanted practical advice and relatable stories. So, 1989 seems right.\n\nComparing the two: Arthur\'s Magazine (1977) vs. First for Women (1989). That makes Arthur\'s Magazine the earlier one by about 12 years. But I need to confirm these dates to be sure. Let me try to recall if there\'s any conflicting information. Another source I remember mentions Arthur\'s Magazine as starting in 1977 and ending in 1985. And First for Women definitely started in the late 80s, 1989. So the conclusion is Arthur\'s Magazine was first. The user might be testing knowledge of magazine history, perhaps for trivia or academic purposes. It\'s possible someone might confuse the names or the time periods, but based on the info I have, Arthur\'s Magazine predates First for Women by a significant margin. I don\'t see any other Arthur\'s Magazines that would overlap here. The key is to make sure there aren\'t other magazines with similar names that could be confused here. For example, the original Arthur\'s Magazine from the 1840s, but I don\'t think that\'s the one being referred to here. The user is likely referring to the DC Comics publication. So the answer is Arthur\'s Magazine started earlier.\n</think>']

for para in paraphrases:
    print(para)
    print("\n------------------------------------------------------\n")

<|im_start|>system
Think through the given question, then output only the final answer.
Output format (strict): ### Answer: <final answer>
Do not include any explanation, reasoning, citations, or extra text—only the single answer line.<|im_end|>
<|im_start|>user
Which magazine was started first Arthur's Magazine or First for Women?<|im_end|>
<|im_start|>assistant
<think>
Okay, I need to figure out which magazine was started first between Arthur's Magazine and First for Women. Let me start by recalling what I know about these publications.

First, Arthur's Magazine. I think there might be a few magazines with similar names. There's The Atlantic Monthly, which was founded in 1857, but that's not Arthur's. Wait, maybe it's Arthur's Magazine from the 19th century? Let me check my memory. I remember that there was an Arthur's Magazine in the 1840s or 50s. Maybe it was a literary or general interest magazine. Let me try to remember. I think Arthur's Magazine was published in the mid-1800s. P

In [13]:
df = pd.read_feather("/home/xzhao/workspace/GYB_self-ensemble/datasets/hotpot-reasoning-debug/qwen3_1.7b/baseline_per_prompt.0shots.0paras.temp1.0.topp0.95.maxnew2048.repeat4.selfparas.feather")

In [15]:
df

,uuid,answers,paraphrase,prompt,generation,is_orig,thinking,messages,generation_lemmas,answer_lemmas
0,5a7a06935542990198eaf050,[Arthur's Magazine],Which magazine was started first Arthur's Maga...,Which magazine was started first Arthur's Maga...,### Answer: Arthur's Magazine,True,"<think>\nOkay, so the question is asking which...",<|im_start|>system\nThink through the given qu...,"[#, #, #, answer, :, arthur, 's, magazine]","[[arthur, 's, magazine]]"
1,5a7a06935542990198eaf050,[Arthur's Magazine],Which magazine was started first Arthur's Maga...,Which magazine was started first Arthur's Maga...,### Answer: Arthur's Magazine,True,"<think>\nOkay, so I need to figure out which m...",<|im_start|>system\nThink through the given qu...,"[#, #, #, answer, :, arthur, 's, magazine]","[[arthur, 's, magazine]]"
2,5a7a06935542990198eaf050,[Arthur's Magazine],Which magazine was started first Arthur's Maga...,Which magazine was started first Arthur's Maga...,### Answer: Arthur's Magazine,True,"<think>\nOkay, let's tackle this question. The...",<|im_start|>system\nThink through the given qu...,"[#, #, #, answer, :, arthur, 's, magazine]","[[arthur, 's, magazine]]"
3,5a7a06935542990198eaf050,[Arthur's Magazine],Which magazine was started first Arthur's Maga...,Which magazine was started first Arthur's Maga...,### Answer: Arthur's Magazine,True,"<think>\nOkay, so the question is asking which...",<|im_start|>system\nThink through the given qu...,"[#, #, #, answer, :, arthur, 's, magazine]","[[arthur, 's, magazine]]"
4,5a7a06935542990198eaf050,[Arthur's Magazine],"Which magazine was established first, Arthur's...","Which magazine was established first, Arthur's...",### Answer: Arthur's Magazine,False,"<think>\nOkay, so I need to figure out which m...",<|im_start|>system\nThink through the given qu...,"[#, #, #, answer, :, arthur, 's, magazine]","[[arthur, 's, magazine]]"
...,...,...,...,...,...,...,...,...,...,...
2315,5a81fab9554299676cceb1b1,[Millard Fillmore],Which American president's father was a farmer...,Which American president's father was a farmer...,### Answer: Grover Cleveland,False,"<think>\nOkay, let's see. The question is aski...",<|im_start|>system\nThink through the given qu...,"[#, #, #, answer, :, grover, cleveland]","[[millard, fillmore]]"
2316,5a81fab9554299676cceb1b1,[Millard Fillmore],Which American president's father was a farmer...,Which American president's father was a farmer...,### Answer: Grover Cleveland,False,"<think>\nOkay, let's try to figure out this qu...",<|im_start|>system\nThink through the given qu...,"[#, #, #, answer, :, grover, cleveland]","[[millard, fillmore]]"
2317,5a81fab9554299676cceb1b1,[Millard Fillmore],Which American president's father was a farmer...,Which American president's father was a farmer...,"<think>\nOkay, let's try to figure out this qu...",False,,<|im_start|>system\nThink through the given qu...,"[<, think, >, \n, okay, ,, let, us, try, to, f...","[[millard, fillmore]]"
2318,5a81fab9554299676cceb1b1,[Millard Fillmore],Which American president's father was a farmer...,Which American president's father was a farmer...,### Answer: Grover Cleveland,False,"<think>\nOkay, let's see. The question is aski...",<|im_start|>system\nThink through the given qu...,"[#, #, #, answer, :, grover, cleveland]","[[millard, fillmore]]"
